# IAQ Control Simulation Results

This notebook visualizes the results from the IAQ control simulation, including:
- CO2 concentration over time
- Zone temperatures
- Occupancy patterns
- Outdoor air multiplier adjustments

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path

## Load Data

In [ ]:
# Load the IAQ control log
# Project root = workspace root (dir that contains 'outputs')
project_root = Path.cwd() if (Path.cwd() / 'outputs').exists() else Path.cwd().parent
log_path = project_root / 'outputs' / 'rl_hvac_control' / 'rl_hvac_log.csv'

if not log_path.exists():
    print(f"Log file not found: {log_path}")
    print("Run the IAQ control simulation first: python tests/test_iaq_control_sim.py")
else:
    df = pd.read_csv(log_path)
    print(f"Loaded {len(df)} timesteps")
    df.head()

Loaded 574 timesteps


In [4]:
# Create a datetime index (assuming 15-minute timesteps starting Jan 1)
df['datetime'] = pd.date_range(start='2024-01-01', periods=len(df), freq='15min')
df['day'] = df['datetime'].dt.dayofyear
df['hour_of_day'] = df['datetime'].dt.hour + df['datetime'].dt.minute / 60
df.head(10)

,episode,timestep,action,reward,energy_cost,comfort_penalty,setpoint_penalty,demand_penalty,total_cost,current_power,...,hour,day,month,co2_Core_bottom,co2_Core_mid,co2_Core_top,co2_Perimeter_mid_ZN_1,co2_Perimeter_mid_ZN_3,datetime,hour_of_day
0,1,1,"[0.9175806045532227, 0.9999927282333374, 0.967...",-4.557277,0.0,4.552277,0.005,0.0,4.557277,0.0,...,0,1,12,400.0,400.0,400.0,400.0,400.0,2024-01-01 00:00:00,0.00
1,1,2,"[0.7380484342575073, 0.9899276494979858, 0.898...",-0.005000,0.0,0.000000,0.005,0.0,0.005000,0.0,...,0,1,12,400.0,400.0,400.0,400.0,400.0,2024-01-01 00:15:00,0.25
2,1,3,"[0.9993005990982056, 0.9917157888412476, -0.93...",-0.005000,0.0,0.000000,0.005,0.0,0.005000,0.0,...,0,1,12,400.0,400.0,400.0,400.0,400.0,2024-01-01 00:30:00,0.50
3,2,1,"[0.9465604424476624, 0.9999977350234985, 0.767...",-0.005000,0.0,0.000000,0.005,0.0,0.005000,0.0,...,0,1,12,400.0,400.0,400.0,400.0,400.0,2024-01-01 00:45:00,0.75
4,2,2,"[0.8790189623832703, 0.9998183846473694, 0.889...",-0.005000,0.0,0.000000,0.005,0.0,0.005000,0.0,...,1,1,12,400.0,400.0,400.0,400.0,400.0,2024-01-01 01:00:00,1.00
5,2,3,"[0.8908429741859436, 0.9981516003608704, 0.923...",-0.005000,0.0,0.000000,0.005,0.0,0.005000,0.0,...,1,1,12,400.0,400.0,400.0,400.0,400.0,2024-01-01 01:15:00,1.25


In [ ]:
# Which episode(s) to plot: int (e.g. 1), list (e.g. [1, 2, 3]), or None for all
EPISODE_TO_PLOT = 1

if EPISODE_TO_PLOT is None:
    plot_df = df.copy()
    print(f"Plotting all episodes ({len(plot_df)} timesteps)")
else:
    if isinstance(EPISODE_TO_PLOT, (list, tuple)):
        plot_df = df[df['episode'].isin(EPISODE_TO_PLOT)].copy()
        print(f"Plotting episodes {EPISODE_TO_PLOT} ({len(plot_df)} timesteps)")
    else:
        plot_df = df[df['episode'] == EPISODE_TO_PLOT].copy()
        print(f"Plotting episode {EPISODE_TO_PLOT} ({len(plot_df)} timesteps)")

# Derive avg_co2_ppm / avg_temp from log columns if missing (for compatibility with plots below)
co2_cols = [c for c in plot_df.columns if c.startswith('co2_')]
if co2_cols and 'avg_co2_ppm' not in plot_df.columns:
    plot_df['avg_co2_ppm'] = plot_df[co2_cols].mean(axis=1)
if 'avg_zone_temp' in plot_df.columns and 'avg_temp' not in plot_df.columns:
    plot_df['avg_temp'] = plot_df['avg_zone_temp']

## CO2 Concentration Analysis

In [5]:
# CO2 over time with thresholds
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df['datetime'],
    y=df['avg_co2_ppm'],
    mode='lines',
    name='Average CO2',
    line=dict(color='#1f77b4', width=1)
))

# Add threshold lines
fig.add_hline(y=800, line_dash='dash', line_color='orange', 
              annotation_text='Target (800 ppm)')
fig.add_hline(y=1000, line_dash='dash', line_color='red', 
              annotation_text='High (1000 ppm)')
fig.add_hline(y=400, line_dash='dot', line_color='green', 
              annotation_text='Outdoor (400 ppm)')

fig.update_layout(
    title='CO2 Concentration Over Time',
    xaxis_title='Date/Time',
    yaxis_title='CO2 (ppm)',
    height=500,
    template='plotly_white'
)

fig.show()

KeyError: 'avg_co2_ppm'

In [ ]:
# CO2 distribution histogram
fig = px.histogram(
    plot_df, x='avg_co2_ppm', nbins=50,
    title='CO2 Concentration Distribution',
    labels={'avg_co2_ppm': 'CO2 (ppm)', 'count': 'Frequency'},
    template='plotly_white'
)

fig.add_vline(x=800, line_dash='dash', line_color='orange', annotation_text='800 ppm')
fig.add_vline(x=1000, line_dash='dash', line_color='red', annotation_text='1000 ppm')

fig.show()

## Daily Patterns

In [ ]:
# Average daily pattern
hourly_avg = plot_df.groupby('hour').agg({
    'avg_co2_ppm': 'mean',
    'avg_temp': 'mean',
    'occupancy_fraction': 'mean',
    'oa_multiplier': 'mean'
}).reset_index()

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('CO2 by Hour', 'Temperature by Hour', 
                    'Occupancy by Hour', 'OA Multiplier by Hour')
)

fig.add_trace(
    go.Bar(x=hourly_avg['hour'], y=hourly_avg['avg_co2_ppm'], 
           name='CO2', marker_color='#1f77b4'),
    row=1, col=1
)

fig.add_trace(
    go.Bar(x=hourly_avg['hour'], y=hourly_avg['avg_temp'], 
           name='Temp', marker_color='#ff7f0e'),
    row=1, col=2
)

fig.add_trace(
    go.Bar(x=hourly_avg['hour'], y=hourly_avg['occupancy_fraction'] * 100, 
           name='Occupancy', marker_color='#2ca02c'),
    row=2, col=1
)

fig.add_trace(
    go.Bar(x=hourly_avg['hour'], y=hourly_avg['oa_multiplier'], 
           name='OA Mult', marker_color='#9467bd'),
    row=2, col=2
)

fig.update_layout(
    height=600, 
    title_text='Average Daily Patterns',
    showlegend=False,
    template='plotly_white'
)

fig.update_xaxes(title_text='Hour of Day')
fig.update_yaxes(title_text='CO2 (ppm)', row=1, col=1)
fig.update_yaxes(title_text='Temperature (°C)', row=1, col=2)
fig.update_yaxes(title_text='Occupancy (%)', row=2, col=1)
fig.update_yaxes(title_text='OA Multiplier', row=2, col=2)

fig.show()

## Multi-Variable Time Series

In [ ]:
# Detailed view: use selected episode(s) data (or first 7 days if plotting all)
week_df = plot_df.copy()
if EPISODE_TO_PLOT is None and 'day' in plot_df.columns:
    week_df = plot_df[plot_df['day'] <= 7].copy() if (plot_df['day'] <= 7).any() else plot_df.copy()

fig = make_subplots(
    rows=4, cols=1,
    shared_xaxes=True,
    subplot_titles=('CO2 Concentration', 'Zone Temperature', 
                    'Occupancy', 'Outdoor Air Multiplier'),
    vertical_spacing=0.08
)

# CO2
fig.add_trace(
    go.Scatter(x=week_df['datetime'], y=week_df['avg_co2_ppm'],
               mode='lines', name='CO2', line=dict(color='#1f77b4')),
    row=1, col=1
)
fig.add_hline(y=800, line_dash='dash', line_color='orange', row=1, col=1)

# Temperature
fig.add_trace(
    go.Scatter(x=week_df['datetime'], y=week_df['avg_temp'],
               mode='lines', name='Temp', line=dict(color='#ff7f0e')),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(x=week_df['datetime'], y=week_df['cooling_sp'],
               mode='lines', name='Cooling SP', line=dict(color='blue', dash='dot')),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(x=week_df['datetime'], y=week_df['heating_sp'],
               mode='lines', name='Heating SP', line=dict(color='red', dash='dot')),
    row=2, col=1
)

# Occupancy
fig.add_trace(
    go.Scatter(x=week_df['datetime'], y=week_df['occupancy_fraction'] * 100,
               mode='lines', name='Occupancy', fill='tozeroy',
               line=dict(color='#2ca02c')),
    row=3, col=1
)

# OA Multiplier
fig.add_trace(
    go.Scatter(x=week_df['datetime'], y=week_df['oa_multiplier'],
               mode='lines', name='OA Mult', line=dict(color='#9467bd')),
    row=4, col=1
)

fig.update_layout(
    height=800,
    title_text='First Week - Detailed View',
    template='plotly_white',
    showlegend=True
)

fig.update_yaxes(title_text='CO2 (ppm)', row=1, col=1)
fig.update_yaxes(title_text='Temp (°C)', row=2, col=1)
fig.update_yaxes(title_text='Occ (%)', row=3, col=1)
fig.update_yaxes(title_text='OA Mult', row=4, col=1)
fig.update_xaxes(title_text='Date/Time', row=4, col=1)

fig.show()

## Correlation Analysis

In [ ]:
# CO2 vs Occupancy scatter
fig = px.scatter(
    plot_df.sample(min(5000, len(plot_df))),  # Sample for performance
    x='occupancy_fraction',
    y='avg_co2_ppm',
    color='oa_multiplier',
    title='CO2 vs Occupancy (colored by OA Multiplier)',
    labels={
        'occupancy_fraction': 'Occupancy Fraction',
        'avg_co2_ppm': 'CO2 (ppm)',
        'oa_multiplier': 'OA Multiplier'
    },
    template='plotly_white',
    opacity=0.5
)

fig.show()

## Summary Statistics

In [ ]:
# Calculate summary statistics (for selected episode(s))
summary = {
    'Total Timesteps': len(plot_df),
    'Simulation Hours': len(plot_df) / 4,
    'Simulation Days': len(plot_df) / 4 / 24,
    '': '',
    'CO2 Min (ppm)': plot_df['avg_co2_ppm'].min(),
    'CO2 Max (ppm)': plot_df['avg_co2_ppm'].max(),
    'CO2 Mean (ppm)': plot_df['avg_co2_ppm'].mean(),
    'CO2 Std (ppm)': plot_df['avg_co2_ppm'].std(),
    ' ': '',
    'Time > 800 ppm (%)': (plot_df['avg_co2_ppm'] > 800).sum() / len(plot_df) * 100,
    'Time > 1000 ppm (%)': (plot_df['avg_co2_ppm'] > 1000).sum() / len(plot_df) * 100,
    '  ': '',
    'Temp Min (°C)': plot_df['avg_temp'].min(),
    'Temp Max (°C)': plot_df['avg_temp'].max(),
    'Temp Mean (°C)': plot_df['avg_temp'].mean(),
}
if 'oa_multiplier' in plot_df.columns:
    summary.update({'   ': '', 'OA Multiplier Min': plot_df['oa_multiplier'].min(), 'OA Multiplier Max': plot_df['oa_multiplier'].max(), 'OA Multiplier Mean': plot_df['oa_multiplier'].mean()})

summary_df = pd.DataFrame(list(summary.items()), columns=['Metric', 'Value'])
summary_df

## Heatmap - CO2 by Hour and Day

In [ ]:
# Create pivot table for heatmap
plot_df['week'] = plot_df['datetime'].dt.isocalendar().week
plot_df['dayofweek'] = plot_df['datetime'].dt.dayofweek

# Aggregate by hour and day of week
heatmap_data = plot_df.groupby(['dayofweek', 'hour'])['avg_co2_ppm'].mean().unstack()

fig = go.Figure(data=go.Heatmap(
    z=heatmap_data.values,
    x=heatmap_data.columns,
    y=['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'],
    colorscale='RdYlGn_r',
    colorbar=dict(title='CO2 (ppm)')
))

fig.update_layout(
    title='Average CO2 by Day of Week and Hour',
    xaxis_title='Hour of Day',
    yaxis_title='Day of Week',
    template='plotly_white',
    height=400
)

fig.show()